# OE Promotional Features — NBA 3.1

**Purpose:** Generate Omnichannel Engine (OE) promotional features for the Kerendia NBA 3.1 model across 28 monthly cohorts.  
**Source:** `oe_kerendia_fact_npp_inference` (NPP table only — FACT_CALL and EVNT_PRSNL_PROM excluded).  
**Reference:** `OE_Promotional_Features_Client_Summary.md` in this folder.

### Features produced (316 = 79 base features x 4 windows P3, P6, P9, P12)
- **Reach / engagement / action (R/I/E) per channel x window**: 22 / 15 / 13 channels. Flags come from `mdb_prep_config.xlsx` (sheet `Metric Mappings`) plus the supplemental Medscape mappings in the Config cell (`EXTRA_METRIC_MAPPINGS`)
- **Content-category counts x window**: 17 tags (`SAMPLES` excluded)
- **Indication R/I/E x window**: HF, CKD, T2D (no T1D records in the data)
- **Touchpoint totals** (exposure, engagement, action) per window
- SFMC HQ Email is excluded (built in notebook 15)

Windows end on each cohort's `lookback_end_date` and go back 3, 6, 9 or 12 months. See the feature dictionary at the end of the notebook.

### Pipeline (self-contained, Spark end to end; no src.NBA.* imports)
1. `npp_processing()`: load NPP, content-tag, derive channel, join R/I/E flags, event-sequence
2. `get_transaction_idc()`: apply +7 day promo lag, assign 18th-to-17th lookback period, dedup
3. `get_counts_long()`: R/I/E, indication and content counts per CUSTOMER_ID x period x feature
4. Cohort assembly: sum periods into P3/P6/P9/P12 per cohort, pivot wide, add touchpoint totals
5. Cross-join with the full HCP x cohort universe, fill 0, QC, write

Set the `test_mode` widget to `true` for a quick 2-period run on cohort c28 (writes `*_TEST.parquet`).

In [0]:
# ── Catalog & Volume paths ───────────────────────────────────────
CATALOG       = "ph-com-ai-us-prod-catalog-2553213839599380"
V31           = f"/Volumes/{CATALOG}/kerendia_nba_3/kerendia_nba_3_1_volume"

# ── Helper files (same OE folder) ────────────────────────────────
HELPER_DIR    = "/Workspace/Users/sehar.aejaz.ext@bayer.com/kerendia_nba_3_1_develop/NBA 3.1 - Dev/04_Promotional_Features/non_personal_promotional_features/OE/helper_files"
CONFIG_FILE   = f"{HELPER_DIR}/mdb_prep_config.xlsx"     # pipeline config
# R/I/E flag lookup: sheet 'Metric Mappings' (Category R/I/E, Event Type = CHANNEL_ID, Mapped Metric = MTR_NM)
FLAG_FILE     = CONFIG_FILE
FLAG_SHEET    = "Metric Mappings"
# Content tagging lookup from original OE workspace (flag if missing)
CONTENT_TAG_FILE = "/Workspace/Users/wenjie.chen@bayer.com/OE/GA Run/Data_for_production/Content_tagging_6_5.xlsx"

# ── Source table ──────────────────────────────────────────────────
NPP_TABLE     = f"`{CATALOG}`.kerendia_oe_journey.oe_kerendia_fact_npp_inference"

# ── NBA 3.1 paths ─────────────────────────────────────────────────
COHORT_PATH   = f"{V31}/cohorts/promotional/cohorts.parquet"
UNIVERSE_PATH = f"{V31}/HCP_Universe_without_call_hcps/"
OUTPUT_PATH   = f"{V31}/notebook_output/oe_nba_3_1/kerendia_nba_3_1_oe_promotional_features.parquet"

# ── ID mapping tables (BH_ID → PROVIDER_ID) ──────────────────────────
MAPPING_TABLE = f"`{CATALOG}`.kerendia_nba_3.SOURCE_PROVIDER_BHID_NPI_MAPPING"
BHID_NPI_OE   = f"`{CATALOG}`.kerendia_oe_journey.kerendia_oe_npi_bhid_mapping_inference"
NPI_RE        = "^[0-9]{10}$"

# ── Processing constants ──────────────────────────────────────────
PROMO_LAG_DAYS  = 7
PRODUCT_FILTER  = "KERENDIA"
# SFMC HQ Email is already built in notebook 15 (HQ Emails) -> excluded here to avoid double counting
EXCLUDE_VENDORS = ["SFMC"]
# Content-tag features not wanted in the output (exposures still count in channel/touchpoint totals)
EXCLUDE_CONTENT_TAGS = ["SAMPLES"]

# Supplemental R/I/E mappings, added to the 'Metric Mappings' sheet at load time.
# These Medscape channels log activity under generic metric names that the sheet does not list.
# Convention (same as the sheet): R = delivery / exposure / impression, I = open / view / time spent,
# E = click; banner clicks count as both I and E (as PULSEPOINT_MEDIA_CLICK in the sheet).
EXTRA_METRIC_MAPPINGS = [
    # (Category, Event Type, Mapped Metric)
    ("R", "MEDSCAPE_BANNER", "EXPOSURE"),            ("R", "MEDSCAPE_BANNER", "IMPRESSIONS"),
    ("I", "MEDSCAPE_BANNER", "LINK CLICK"),          ("I", "MEDSCAPE_BANNER", "CLICKED"),
    ("E", "MEDSCAPE_BANNER", "LINK CLICK"),          ("E", "MEDSCAPE_BANNER", "CLICKED"),
    ("R", "MEDSCAPE_PULSEPOINT_BANNER", "EXPOSURE"), ("R", "MEDSCAPE_PULSEPOINT_BANNER", "IMPRESSIONS"),
    ("I", "MEDSCAPE_PULSEPOINT_BANNER", "LINK CLICK"), ("I", "MEDSCAPE_PULSEPOINT_BANNER", "CLICKED"),
    ("E", "MEDSCAPE_PULSEPOINT_BANNER", "LINK CLICK"), ("E", "MEDSCAPE_PULSEPOINT_BANNER", "CLICKED"),
    # BrandSpot rich media: PARTICIPATION is the pre-Aug-2025 exposure record (as RICH_MEDIA_ENGAGEMENT = R in the sheet)
    ("R", "MEDSCAPE_BRANDSPOT", "EXPOSURE"),         ("R", "MEDSCAPE_BRANDSPOT", "IMPRESSIONS"),
    ("R", "MEDSCAPE_BRANDSPOT", "PARTICIPATION"),
    ("I", "MEDSCAPE_BRANDSPOT", "VIDEO TIME SPENT"),
    ("E", "MEDSCAPE_BRANDSPOT", "LINK CLICK"),       ("E", "MEDSCAPE_BRANDSPOT", "CLICKED"),
    # DDTRD and FAP: email + content programmes (VIEW = R and I, as PAGE_VIEW for BrandAlert in the sheet)
    ("R", "MEDSCAPE_DDTRD", "DELIVERED"),  ("R", "MEDSCAPE_DDTRD", "EXPOSURE"), ("R", "MEDSCAPE_DDTRD", "VIEW"),
    ("I", "MEDSCAPE_DDTRD", "OPENED"),     ("I", "MEDSCAPE_DDTRD", "VIEW"),
    ("E", "MEDSCAPE_DDTRD", "CLICKED"),    ("E", "MEDSCAPE_DDTRD", "PARTICIPATION"),
    ("R", "MEDSCAPE_FAP", "DELIVERED"),    ("R", "MEDSCAPE_FAP", "EXPOSURE"),   ("R", "MEDSCAPE_FAP", "VIEW"),
    ("I", "MEDSCAPE_FAP", "OPENED"),       ("I", "MEDSCAPE_FAP", "VIEW"),
    ("E", "MEDSCAPE_FAP", "CLICKED"),      ("E", "MEDSCAPE_FAP", "LINK CLICK"),
]

# ── Test mode (default false = full production run) ──────────────────
# true -> only the last TEST_MONTHS of NPP data and TEST_COHORTS; output written to *_TEST.parquet
dbutils.widgets.dropdown("test_mode", "false", ["true", "false"], "Test mode")
TEST_MODE    = dbutils.widgets.get("test_mode") == "true"
TEST_MONTHS  = 2
TEST_COHORTS = ["c28"]          # latest cohort; its C3 window covers the last 2 data months
if TEST_MODE:
    OUTPUT_PATH = OUTPUT_PATH.replace(".parquet", "_TEST.parquet")
    print(f"*** TEST MODE: {TEST_MONTHS} months, cohorts {TEST_COHORTS} ***")
INDICATIONS     = ["HF", "CKD", "T1D", "T2D"]

print("Config loaded.")
print(f"  NPP source : {NPP_TABLE}")
print(f"  Output     : {OUTPUT_PATH}")

In [0]:
import os, re, sys
import pandas as pd
import numpy as np
from datetime import datetime
from functools import reduce
from dateutil.relativedelta import relativedelta

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.functions import col, to_date, when, lit, upper, trim
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

# ── Utility helpers (replicate src.NBA.utils inline) ─────────────

def standardize_col_values(series: pd.Series) -> pd.Series:
    """Upper-case and replace non-alphanumeric chars with underscores."""
    return (series.astype(str).str.upper()
                               .str.strip()
                               .str.replace(r'[^A-Z0-9]+', '_', regex=True)
                               .str.strip('_'))

def clean_spark_cols(sdf, cols, ph="_"):
    """Trim, null-fill with placeholder, normalise special chars."""
    bad = ["", " ", "None", "none", "NaN", "nan", "NULL", "null", "Null", "N/A", "n/a", "NA"]
    for c in cols:
        sdf = (sdf
               .withColumn(c, F.trim(F.col(c).cast("string")))
               .withColumn(c, when(F.col(c).isNull() | F.col(c).isin(bad), F.lit(ph)).otherwise(F.col(c)))
               .withColumn(c, F.regexp_replace(F.col(c), r'[^\w]+', '_'))
               .withColumn(c, F.regexp_replace(F.col(c), r'_+', '_'))
               .withColumn(c, F.regexp_replace(F.col(c), r'^_+|_+$', ''))
               .withColumn(c, when(F.col(c).isNull() | (F.col(c) == ""), F.lit(ph)).otherwise(F.col(c))))
    return sdf

print("Imports and helpers ready.")

In [0]:
%pip install openpyxl -q

In [0]:
import os

# ── 1. R/I/E flag lookup (mdb_prep_config.xlsx, sheet 'Metric Mappings') ──
# Columns: Category (R/I/E), Event Type (= VEND_NM_CHNL_NM_NEW), Mapped Metric (= MTR_NM)
# One (channel, metric) pair can carry several categories (e.g. MEDSCAPE_FFQ PARTICIPATION = I and E).
_flag_ok = os.path.exists(FLAG_FILE)
if _flag_ok:
    _mm = pd.read_excel(FLAG_FILE, sheet_name=FLAG_SHEET)
    _mm = pd.concat([_mm[['Category', 'Event Type', 'Mapped Metric']],
                     pd.DataFrame(EXTRA_METRIC_MAPPINGS, columns=['Category', 'Event Type', 'Mapped Metric'])],
                    ignore_index=True)
    print(f"Supplemental mappings added: {len(EXTRA_METRIC_MAPPINGS)} rows "
          f"({sorted({e for _, e, _ in EXTRA_METRIC_MAPPINGS})})")
    print(f"Flag mapping loaded: {len(_mm):,} rows | {_mm['Category'].value_counts().to_dict()}")
    _std = lambda s: s.astype(str).str.upper().str.strip().str.replace(r'[^A-Z0-9]+', '_', regex=True)
    _mm['CH_KEY']  = _std(_mm['Event Type'])
    _mm['MTR_KEY'] = _std(_mm['Mapped Metric'])
    _mm['Category'] = _mm['Category'].astype(str).str.upper().str.strip()
    df_flag_pd = (_mm.assign(**{'Exposure Flag':   (_mm['Category'] == 'R').astype(int),
                                'Engagement Flag': (_mm['Category'] == 'I').astype(int),
                                'Acted Flag':      (_mm['Category'] == 'E').astype(int)})
                     .groupby(['CH_KEY','MTR_KEY'], as_index=False)
                     [['Exposure Flag','Engagement Flag','Acted Flag']].max())
    print(f"  Distinct (channel, metric) keys: {len(df_flag_pd):,}")
    df_flag_spark = spark.createDataFrame(df_flag_pd)
else:
    df_flag_spark = None
    print(f"⚠  FLAG FILE NOT FOUND: {FLAG_FILE}")
    print("   All NPP rows will default to Exposure Flag=1 (R only, no I or E features).")

# ── 2. Content tagging lookup ─────────────────────────────────────
# Expected: 14 merge columns + 'Content Category'
_tagg_ok = os.path.exists(CONTENT_TAG_FILE)
if _tagg_ok:
    df_tagg_pd = pd.read_excel(CONTENT_TAG_FILE)
    df_tagg_pd['LITCODE'] = df_tagg_pd['LITCODE'].astype(str)
    print(f"\nContent tagging file loaded: {df_tagg_pd.shape[0]:,} rows")
    print(f"  Content categories: {sorted(df_tagg_pd['Content Category'].dropna().unique())}")
    df_tagg_spark = spark.createDataFrame(df_tagg_pd)
else:
    df_tagg_spark = None
    print(f"\n⚠  CONTENT TAGGING FILE NOT FOUND: {CONTENT_TAG_FILE}")
    print("   All rows will get CONTENT_TAG = 'GENERAL' (no content-category features).")

# ── 3. Inspect mdb_prep_config ────────────────────────────────────
if os.path.exists(CONFIG_FILE):
    xl  = pd.ExcelFile(CONFIG_FILE)
    print(f"\nConfig sheets: {xl.sheet_names}")
else:
    print(f"\n⚠  CONFIG FILE NOT FOUND: {CONFIG_FILE}")

In [0]:
# ── Cohorts ───────────────────────────────────────────────────────
cohorts_df = spark.read.parquet(COHORT_PATH).toPandas()
# Normalise column names (parquet uses lowercase snake_case)
cohorts_df = cohorts_df.rename(columns={
    'cohort_id': 'COHORT', 'lookback_end_date': 'LOOKBACK_END_DATE',
    'dataset': 'DATASET', 'prediction_month': 'PREDICTION_MONTH'
})
cohorts_df['LOOKBACK_END_DATE'] = pd.to_datetime(cohorts_df['LOOKBACK_END_DATE'])
if TEST_MODE:
    cohorts_df = cohorts_df[cohorts_df['COHORT'].isin(TEST_COHORTS)].reset_index(drop=True)
print(f"Cohorts: {len(cohorts_df)} rows")
display(cohorts_df[['COHORT','LOOKBACK_END_DATE','DATASET']].head(30))

# ── HCP Universe ──────────────────────────────────────────────────
hcp_spark = spark.read.parquet(UNIVERSE_PATH)
N_HCP = hcp_spark.select('PROVIDER_ID').distinct().count()
print(f"\nHCP Universe: {N_HCP:,} PROVIDER_IDs")

# ── Full PROVIDER_ID × COHORT grid (for zero-fill later) ──────────
# (the grid itself is built in Spark in the cross-join step; no pandas copy is kept)
print(f"Full grid: {N_HCP * len(cohorts_df):,} rows ({N_HCP:,} HCPs × {len(cohorts_df)} cohorts)")

In [0]:
# ── Lookback periods ─────────────────────────────────────────────
# Every LOOKBACK_END_DATE falls on the same day of month (the 17th). Each event is assigned
# to a "period" YRMO = the month whose (CUTOFF_DAY+1 of previous month .. CUTOFF_DAY] interval
# contains it, e.g. 2026-08-18 .. 2026-09-17 -> period 202609.
# Window Pn for a cohort = the n periods ending at its LOOKBACK_END_DATE (inclusive).
_days = cohorts_df['LOOKBACK_END_DATE'].dt.day.unique()
assert len(_days) == 1, f"LOOKBACK_END_DATE days differ across cohorts: {_days}"
CUTOFF_DAY = int(_days[0])

earliest_anchor = cohorts_df['LOOKBACK_END_DATE'].min()
latest_anchor   = cohorts_df['LOOKBACK_END_DATE'].max()

# Period range needed: P12 of the earliest cohort .. the latest cohort's end period
GLOBAL_END_YRMO   = int(latest_anchor.strftime('%Y%m'))
GLOBAL_START_YRMO = int((earliest_anchor.replace(day=1) - relativedelta(months=11)).strftime('%Y%m'))
if TEST_MODE:   # keep only the last TEST_MONTHS periods
    _ts = (pd.Timestamp(year=GLOBAL_END_YRMO // 100, month=GLOBAL_END_YRMO % 100, day=1)
           - relativedelta(months=TEST_MONTHS - 1))
    GLOBAL_START_YRMO = int(_ts.strftime('%Y%m'))
# Raw NPP must start one calendar month earlier (period YYYYMM starts on the 18th of the prior month)
NPP_LOAD_START_YRMO = int((pd.Timestamp(year=GLOBAL_START_YRMO // 100, month=GLOBAL_START_YRMO % 100, day=1)
                           - relativedelta(months=1)).strftime('%Y%m'))

_p0 = (pd.Timestamp(year=GLOBAL_START_YRMO // 100, month=GLOBAL_START_YRMO % 100, day=1)
       - relativedelta(months=1)).replace(day=CUTOFF_DAY) + pd.Timedelta(days=1)
print(f"Earliest anchor : {earliest_anchor.date()}")
print(f"Latest anchor   : {latest_anchor.date()}")
print(f"Period range    : {GLOBAL_START_YRMO} → {GLOBAL_END_YRMO}  "
      f"(event dates {_p0.date()} .. {latest_anchor.date()}, after promo lag)")

# ── Build BH_ID → PROVIDER_ID map for the NBA 3.1 universe ───────────────
# NPP uses BH_ID (CUST_ID); NBA 3.1 universe uses PROVIDER_ID — need a bridge
_univ  = hcp_spark.select("PROVIDER_ID").distinct()

_m_bh  = (spark.table(MAPPING_TABLE)
           .select(F.trim("BH_ID").alias("BH_ID"), "PROVIDER_ID")
           .filter(col("BH_ID").isNotNull() & col("PROVIDER_ID").isNotNull())
           .join(_univ, "PROVIDER_ID", "left_semi"))

_m_npi = (spark.table(MAPPING_TABLE)
           .select(F.trim("NPI_NUMBER").alias("NPI_NUMBER"), "PROVIDER_ID")
           .filter(col("NPI_NUMBER").rlike(NPI_RE) & col("PROVIDER_ID").isNotNull())
           .join(_univ, "PROVIDER_ID", "left_semi"))

_bh_oe = (spark.table(BHID_NPI_OE)
           .select(F.trim("BH_ID").alias("BH_ID"), F.trim("NPI_NUMBER").alias("NPI_NUMBER"))
           .filter(col("BH_ID").isNotNull() & col("NPI_NUMBER").rlike(NPI_RE)).distinct()
           .join(_m_npi, "NPI_NUMBER")
           .groupBy("BH_ID").agg(F.min("PROVIDER_ID").alias("PROVIDER_ID")))

# Primary + OE-fallback; keep one PROVIDER_ID per BH_ID (min)
BH_ID_MAP = (_m_bh.select("BH_ID", "PROVIDER_ID")
             .unionByName(_bh_oe.select("BH_ID", "PROVIDER_ID"))
             .groupBy("BH_ID").agg(F.min("PROVIDER_ID").alias("PROVIDER_ID")))

# df_target_spark: just BH_ID column for the npp_processing() inner join
df_target_spark = BH_ID_MAP.select("BH_ID").distinct()
print(f"BH_ID → PROVIDER_ID map: {BH_ID_MAP.count():,} entries")
print(f"Target BH_IDs for NPP join: {df_target_spark.count():,}")

In [0]:
def npp_processing(spark, npp_table, df_flag_spark, df_tagg_spark,
                   start_yrmo, end_yrmo, df_target_spark,
                   promo_lag_days=7, product_filter="KERENDIA", exclude_vendors=()):
    """
    Load & process NPP (Non-Personal Promotions) for Kerendia.
    Equivalent to Transactions.email_processing() in data_processing.py,
    renamed to npp_processing() per OE_Promotional_Features_Client_Summary.md.

    Returns a Spark DataFrame with 9 columns:
      CUSTOMER_ID, CHANNEL_ID, CONTENT_TAG, PRODUCT_ID, INTERACTION_ID,
      EXPOSED_ON, ENGAGED_ON, ACTED_ON, INDICATION
    """
    # Date boundaries (promo lag applied BACKWARD so raw data covers lag-shifted window)
    start_bnd = (pd.Timestamp(str(start_yrmo), format=None)
                   if isinstance(start_yrmo, str)
                   else pd.Timestamp(year=start_yrmo // 100, month=start_yrmo % 100, day=1))
    start_bnd = (start_bnd - pd.Timedelta(days=promo_lag_days)).date()
    end_bnd   = ((pd.Timestamp(year=end_yrmo // 100, month=end_yrmo % 100, day=1)
                  + pd.offsets.MonthEnd(0)) - pd.Timedelta(days=promo_lag_days)).date()

    # ── Load & filter ────────────────────────────────────────────
    df = spark.table(npp_table)
    df = (df
          .filter((col("PROD_BRAND_NM") == product_filter) & col("BAYER_HCP_ID").isNotNull())
          .filter(~F.upper(F.trim(col("VEND_NM"))).isin([v.upper() for v in exclude_vendors]))
          .withColumn("DATE_ID_DT", to_date(col("DATE_ID").cast("string"), "yyyyMMdd"))
          .filter(col("DATE_ID_DT").between(lit(start_bnd), lit(end_bnd)))
          .join(df_target_spark, df["CUST_ID"] == df_target_spark["BH_ID"], "inner")
          .dropDuplicates(["UNIQ_NPP_ID","DATE_ID","CUST_ID","VEND_NM","CHNL_NM","MTR_NM"]))
    print(f"NPP after filter + dedup: {df.count():,} rows")

    # Preserve CF_INDICATION for indication features
    df = df.withColumn("CF_INDICATION_orig", F.upper(F.trim(F.col("CF_INDICATION").cast("string"))))

    # ── Content tagging ──────────────────────────────────────────
    merge_cols = ['CHNL_NM','VEND_NM','LITCODE','CNTT_NM','EMAIL_SUBJ_LN','CRTV_NM',
                  'TACT_NM','CPGN_NM','PRMRY_KEY_MSG','CF_NAME','CF_INDICATION',
                  'RPT_CHNL','CHNL_TYP','TACT_TYP']
    # Save originals before cleaning
    for c in ['VEND_NM','CHNL_NM','MTR_NM','TACT_NM']:
        df = df.withColumn(f"{c}_orig", F.col(c))

    if df_tagg_spark is not None:
        df      = clean_spark_cols(df,         merge_cols)
        df_tagg = clean_spark_cols(df_tagg_spark, merge_cols).dropDuplicates(merge_cols)
        df      = (df.join(df_tagg, on=merge_cols, how="left")
                     .withColumn("Content Category",
                                 when(F.col("Content Category").isin("-","Not Needed")
                                      | F.col("Content Category").isNull(), F.lit("General"))
                                 .otherwise(F.col("Content Category")))
                     .withColumnRenamed("Content Category", "CONTENT_TAG"))
    else:
        df = df.withColumn("CONTENT_TAG", F.lit("General"))

    # Restore originals
    for c in ['VEND_NM','CHNL_NM','MTR_NM','TACT_NM']:
        df = df.withColumn(c, F.col(f"{c}_orig")).drop(f"{c}_orig")

    # ── Derive CHNL_NM_NEW ────────────────────────────────────────
    _t = F.upper(F.col('TACT_NM'))
    _p = F.upper(F.regexp_extract(F.col('TACT_NM'), r'(?i)(?:KERENDIA|Kerendiafinerenone)_([A-Za-z]+)', 1))
    df = df.withColumn('CHNL_NM_NEW',
        when(F.col('VEND_NM')=='DOXIMITY',
             when(_t.contains('DOCNEWS'), 'DocNews').when(_t.contains('DOCSPOT'), 'DocSpot')
            .when(_t.contains('DOCPOINT'), 'DocPoint').when(_t.contains('FORMULARY'), 'FormularyOne')
            .when(_t.contains('ANNOUNCE'), 'Announcement').when(_t.rlike(r'\bDN\b'), 'DocNews')
            .when(_t.rlike(r'\bF1\b'), 'FormularyOne').when(_t.rlike(r'\bAN\b'), 'Announcement'))
        .when(F.col('VEND_NM')=='MEDSCAPE',
             when(_p=='EA','3rd Party Email').when(_p=='BA','BrandAlert').when(_p=='BS','BrandSpot')
            .when(_p=='PP','PulsePoint Banner').when(_p=='FFQ','FFQ').when(_p=='RECAP','RECAP')
            .when(_p=='TLM','BANNER').when(_p=='PPEHR','EHR').when(_p=='FAP','FAP').when(_p=='DDTRD','DDTRD'))
        .when(F.col('VEND_NM')=='PEERDIRECT',
             when(_t.contains('MDFI - EMAIL'),'MDFI - Email').when(_t.contains('MDFI - WEB'),'MDFI - Web')
            .when(_t.contains('MDFI - BANNER'),'MDFI - Banner')
            .when(_t.contains('MD MINUTE - EMAIL'),'MD Minute - Email')
            .when(_t.contains('MD MINUTE - WEB'),'MD Minute - Web')
            .when(_t.contains('MD MINUTE - BANNER'),'MD Minute - Banner'))
        .when(F.col('VEND_NM')=='SFMC',       F.lit('HQ Email'))
        .when(F.col('VEND_NM')=='PULSEPOINT', F.lit('Banner')))

    # ── R/I/E flag join ───────────────────────────────────────────
    # Join key: standardised VEND_NM_CHNL_NM_NEW (= 'Event Type') and MTR_NM (= 'Mapped Metric').
    # Rows whose (channel, metric) is not in the mapping get no flag and are dropped below (OE rule).
    if df_flag_spark is not None:
        df = (df.withColumn('CH_KEY',  F.regexp_replace(F.upper(F.concat_ws('_', 'VEND_NM', 'CHNL_NM_NEW')), '[^A-Z0-9]+', '_'))
                .withColumn('MTR_KEY', F.regexp_replace(F.upper(F.trim('MTR_NM')), '[^A-Z0-9]+', '_'))
                .join(F.broadcast(df_flag_spark), on=['CH_KEY','MTR_KEY'], how='left')
                .withColumn('Exposure Flag',   when(F.col('Exposure Flag')   == 1, 1))
                .withColumn('Engagement Flag', when(F.col('Engagement Flag') == 1, 1))
                .withColumn('Acted Flag',      when(F.col('Acted Flag')      == 1, 1))
                .drop('CH_KEY','MTR_KEY'))
    else:
        df = (df.withColumn('Exposure Flag',  F.lit(1))
                .withColumn('Engagement Flag', F.lit(None).cast('int'))
                .withColumn('Acted Flag',       F.lit(None).cast('int')))

    # Keep only matched & valid rows
    df = (df.filter(F.col('CHNL_NM_NEW').isNotNull() &
                    (F.col('Exposure Flag').isNotNull() | F.col('Engagement Flag').isNotNull()
                     | F.col('Acted Flag').isNotNull()))
            .na.drop(subset=['UNIQ_NPP_ID']))

    base = ['CUST_ID','VEND_NM','CHNL_NM_NEW','CONTENT_TAG','PROD_BRAND_NM',
            'UNIQ_NPP_ID','Exposure Flag','Engagement Flag','Acted Flag',
            'DATE_ID_DT','CF_INDICATION_orig']
    df = df.select(base).dropDuplicates()

    # ── Event sequencing ─────────────────────────────────────────
    # One exposure per interaction (first exposure date). Engagement / action are attached to
    # that exposure at most once each (first date), so repeated opens or clicks (same day or
    # later days) count once and engagement can never exceed exposure. Engagement or action
    # records with no exposure for the interaction are dropped.
    keys = ['CUST_ID','VEND_NM','CHNL_NM_NEW','CONTENT_TAG','PROD_BRAND_NM','UNIQ_NPP_ID']

    exp_df = (df.filter(F.col('Exposure Flag')==1).groupBy(*keys)
                .agg(F.min('DATE_ID_DT').alias('EXPOSED_ON'),
                     F.min_by('CF_INDICATION_orig', 'DATE_ID_DT').alias('INDICATION')))
    eng_df = (df.filter(F.col('Engagement Flag')==1).groupBy(*keys)
                .agg(F.min('DATE_ID_DT').alias('ENGAGED_ON')))
    act_df = (df.filter(F.col('Acted Flag')==1).groupBy(*keys)
                .agg(F.min('DATE_ID_DT').alias('ACTED_ON')))

    result = (exp_df.join(eng_df, keys, 'left')
                    .join(act_df, keys, 'left')
                    .withColumn('CHANNEL_ID', F.concat_ws('_', F.col('VEND_NM'), F.col('CHNL_NM_NEW')))
                    .withColumnRenamed('CUST_ID',       'CUSTOMER_ID')
                    .withColumnRenamed('PROD_BRAND_NM', 'PRODUCT_ID')
                    .withColumnRenamed('UNIQ_NPP_ID',   'INTERACTION_ID')
                    .select('CUSTOMER_ID','CHANNEL_ID','CONTENT_TAG','PRODUCT_ID',
                            'INTERACTION_ID','EXPOSED_ON','ENGAGED_ON','ACTED_ON','INDICATION')
                    .dropDuplicates(['CUSTOMER_ID','CHANNEL_ID','CONTENT_TAG','INTERACTION_ID']))

    # Kept as a Spark DataFrame: converting the full history to pandas ran out of memory
    return result

print("npp_processing() defined.")

In [0]:
def period_yrmo_col(c, cutoff_day=None):
    """Lookback period (YYYYMM int) of a date column: day <= cutoff -> same month, else next month.
    With cutoff 17: 2026-09-17 -> 202609, 2026-09-18 -> 202610
    (so period 202609 = event dates 2026-08-18 .. 2026-09-17)."""
    cutoff_day = CUTOFF_DAY if cutoff_day is None else cutoff_day
    d = F.col(c) if isinstance(c, str) else c
    return F.date_format(when(F.dayofmonth(d) <= cutoff_day, d).otherwise(F.add_months(d, 1)),
                         'yyyyMM').cast('int')


def _std_name(c):
    """Upper-case, non-alphanumerics -> '_', strip leading/trailing '_'."""
    return F.regexp_replace(F.regexp_replace(F.upper(F.trim(c)), '[^A-Z0-9]+', '_'), '^_+|_+$', '')


def _feat_name(c):
    """Feature column name: standardised and prefixed with OE_."""
    n = _std_name(c)
    return when(n.startswith('OE_'), n).otherwise(F.concat(lit('OE_'), n))


def get_transaction_idc(proto_idc, start_yrmo, end_yrmo, promo_lag_days=7):
    """
    Post-process npp_processing() output (Spark):
      - +promo_lag_days on EXPOSED_ON / ENGAGED_ON / ACTED_ON
      - YRMO = lookback period of EXPOSED_ON; keep periods in [start_yrmo, end_yrmo]
      - standardise CHANNEL_ID and CONTENT_TAG (missing/none -> GENERAL)
      - one row per CUSTOMER_ID x INTERACTION_ID x CHANNEL_ID x EXPOSED_ON; COUNT = 1
    """
    df = proto_idc
    for c in ['EXPOSED_ON', 'ENGAGED_ON', 'ACTED_ON']:
        df = df.withColumn(c, F.date_add(F.to_date(F.col(c)), promo_lag_days))
    df = (df.withColumn('YRMO', period_yrmo_col('EXPOSED_ON'))
            .filter(F.col('YRMO').between(start_yrmo, end_yrmo))
            .withColumn('CHANNEL_ID', _std_name(F.col('CHANNEL_ID').cast('string')))
            .withColumn('CONTENT_TAG', F.upper(F.trim(F.coalesce(F.col('CONTENT_TAG'), lit('GENERAL')))))
            .withColumn('CONTENT_TAG', when(F.col('CONTENT_TAG').isin('NONE', 'NAN', '', 'OTHERS', 'NONE_'),
                                            lit('GENERAL')).otherwise(F.col('CONTENT_TAG'))))
    w = (Window.partitionBy('CUSTOMER_ID', 'INTERACTION_ID', 'CHANNEL_ID', 'EXPOSED_ON')
               .orderBy('CONTENT_TAG', 'ENGAGED_ON', 'ACTED_ON', 'INDICATION'))
    return (df.select('CUSTOMER_ID', 'PRODUCT_ID', 'CHANNEL_ID', 'CONTENT_TAG', 'EXPOSED_ON',
                      'ENGAGED_ON', 'ACTED_ON', 'INTERACTION_ID', 'YRMO', 'INDICATION')
              .dropDuplicates()
              .withColumn('_rn', F.row_number().over(w))
              .filter(F.col('_rn') == 1).drop('_rn')
              .withColumn('COUNT', lit(1)))


def get_counts_long(tx, indications, exclude_content_tags=()):
    """
    Long feature table: one row per CUSTOMER_ID x YRMO (lookback period) x FEATURE with CNT.
      OE_{CHANNEL}_R / _I / _E : exposures / engaged exposures / acted exposures, all counted in
                                 the EXPOSED_ON period (so I <= R and E <= R in every window)
      OE_IND_{ind}_R / _I / _E : same, restricted to rows whose INDICATION contains ind
      OE_{CONTENT_TAG}_COUNT   : exposures by content tag (EXPOSED_ON period)
    """
    parts = []
    ind_u = F.upper(F.coalesce(F.col('INDICATION'), lit('UNKNOWN')))
    for date_col, sfx in [('EXPOSED_ON', 'R'), ('ENGAGED_ON', 'I'), ('ACTED_ON', 'E')]:
        base = tx.filter(F.col(date_col).isNotNull())   # YRMO stays = exposure period
        parts.append(base.select('CUSTOMER_ID', 'YRMO',
                                 _feat_name(F.concat_ws('_', 'CHANNEL_ID', lit(sfx))).alias('FEATURE'), 'COUNT'))
        for ind in indications:
            parts.append(base.filter(ind_u.contains(ind.upper()))
                             .select('CUSTOMER_ID', 'YRMO',
                                     lit(f'OE_IND_{ind.upper()}_{sfx}').alias('FEATURE'), 'COUNT'))
    parts.append(tx.select('CUSTOMER_ID', 'YRMO',
                           _feat_name(F.concat_ws('_', 'CONTENT_TAG', lit('COUNT'))).alias('FEATURE'), 'COUNT'))

    out = reduce(lambda a, b: a.unionByName(b), parts)
    _excl = [f"OE_{re.sub(r'[^A-Z0-9]+', '_', t.upper()).strip('_')}_COUNT" for t in exclude_content_tags]
    if _excl:
        out = out.filter(~F.col('FEATURE').isin(_excl))
    return out.groupBy('CUSTOMER_ID', 'YRMO', 'FEATURE').agg(F.sum('COUNT').alias('CNT'))

print("Spark helpers defined.")

In [0]:
# Intermediate parquet (avoids recomputing the 68M-row NPP pipeline in every later step)
INTERMEDIATE_DIR = OUTPUT_PATH.rsplit('/', 1)[0] + ('/_intermediate_test' if TEST_MODE else '/_intermediate')

# ── Step 1: Load & process NPP ───────────────────────────────────
proto_idc = npp_processing(
    spark          = spark,
    npp_table      = NPP_TABLE,
    df_flag_spark  = df_flag_spark,
    df_tagg_spark  = df_tagg_spark,
    start_yrmo     = NPP_LOAD_START_YRMO,   # one month earlier so the first period is complete
    end_yrmo       = GLOBAL_END_YRMO,
    df_target_spark= df_target_spark,
    promo_lag_days = PROMO_LAG_DAYS,
    product_filter = PRODUCT_FILTER,
    exclude_vendors= EXCLUDE_VENDORS,
)

# ── Step 2: Post-process (lag, period, standardise, dedup) ─────────────
(get_transaction_idc(proto_idc, GLOBAL_START_YRMO, GLOBAL_END_YRMO, PROMO_LAG_DAYS)
    .write.mode('overwrite').parquet(f"{INTERMEDIATE_DIR}/transactions.parquet"))
transactions_spk = spark.read.parquet(f"{INTERMEDIATE_DIR}/transactions.parquet")

_s = transactions_spk.agg(F.count('*').alias('rows'), F.countDistinct('CUSTOMER_ID').alias('hcps'),
                          F.min('YRMO').alias('p0'), F.max('YRMO').alias('p1')).collect()[0]
print(f"Transactions: {_s['rows']:,} rows | {_s['hcps']:,} HCPs | periods {_s['p0']} -> {_s['p1']}")
print("Channels    :", sorted(r[0] for r in transactions_spk.select('CHANNEL_ID').distinct().collect()))
print("Content tags:", sorted(r[0] for r in transactions_spk.select('CONTENT_TAG').distinct().collect()))
print("CF_INDICATION distribution (top 15):")
display(transactions_spk.groupBy(F.upper(F.coalesce('INDICATION', lit('UNKNOWN'))).alias('INDICATION'))
                        .count().orderBy(F.desc('count')).limit(15))

# ── Step 3: R/I/E, indication and content counts (long format) ─────────
(get_counts_long(transactions_spk, INDICATIONS, EXCLUDE_CONTENT_TAGS)
    .write.mode('overwrite').parquet(f"{INTERMEDIATE_DIR}/feature_counts_long.parquet"))
counts_long = spark.read.parquet(f"{INTERMEDIATE_DIR}/feature_counts_long.parquet")

BASE_FEATURES = sorted(r[0] for r in counts_long.select('FEATURE').distinct().collect())
_ch = [f for f in BASE_FEATURES if not f.startswith('OE_IND_') and not f.endswith('_COUNT')]
print(f"\nBase features: {len(BASE_FEATURES)} | channel R/I/E: "
      f"{sum(f.endswith('_R') for f in _ch)}/{sum(f.endswith('_I') for f in _ch)}/{sum(f.endswith('_E') for f in _ch)} | "
      f"indication: {sum(f.startswith('OE_IND_') for f in BASE_FEATURES)} | "
      f"content: {sum(f.endswith('_COUNT') for f in BASE_FEATURES)}")
_no_ind = [i for i in INDICATIONS if not any(f.startswith(f'OE_IND_{i}_') for f in BASE_FEATURES)]
if _no_ind:
    print(f"No data for indications: {_no_ind}")
print("\nAll intermediate tables ready. Proceeding to cohort assembly.")

In [0]:
# ── Build period -> (COHORT, WINDOW) map ────────────────────────────────────
# Pn = the n lookback periods ending at LOOKBACK_END_DATE, i.e. event dates in
# (LOOKBACK_END_DATE - n months, LOOKBACK_END_DATE]. Example c28 (end 2026-09-17):
#   P3 = 2026-06-18..2026-09-17, P12 = 2025-09-18..2026-09-17
WINDOW_MONTHS = {'P3': 3, 'P6': 6, 'P9': 9, 'P12': 12}
WINDOWS       = list(WINDOW_MONTHS)
_yrmo_rows = []
for _, _r in cohorts_df.iterrows():
    _end = pd.Timestamp(_r['LOOKBACK_END_DATE']).replace(day=1)   # end period
    for _w, _n in WINDOW_MONTHS.items():
        for _i in range(_n):
            _yrmo_rows.append({'COHORT': _r['COHORT'], 'WINDOW': _w,
                               'YRMO': int((_end - relativedelta(months=_i)).strftime('%Y%m'))})
yrmo_map_spk = spark.createDataFrame(pd.DataFrame(_yrmo_rows))
print(f"Period-window map: {len(_yrmo_rows)} rows")

# ── Sum counts per CUSTOMER_ID x COHORT x FEATURE x WINDOW, then pivot wide ──
_fw_order = [f"{f}_{w}" for f in BASE_FEATURES for w in WINDOWS]
_lw = (counts_long.join(F.broadcast(yrmo_map_spk), on='YRMO', how='inner')
                  .groupBy('CUSTOMER_ID', 'COHORT', 'FEATURE', 'WINDOW').agg(F.sum('CNT').alias('CNT'))
                  .withColumn('FW', F.concat_ws('_', 'FEATURE', 'WINDOW')))
feat_wide = (_lw.groupBy('CUSTOMER_ID', 'COHORT').pivot('FW', _fw_order).agg(F.first('CNT'))
                .select('CUSTOMER_ID', 'COHORT',
                        *[F.coalesce(F.col(c), lit(0.0)).cast('double').alias(c) for c in _fw_order]))

# ── Touchpoint rollups: sum of channel R / I / E cols per window ──────────
# (indication cols are excluded because they also end in _R_/_I_/_E_<w>)
for _w in WINDOWS:
    for _sfx, _name in [('R', 'OE_EXPOSURE_TOUCHPOINT'), ('I', 'OE_ENGAGEMENT_TOUCHPOINT'),
                        ('E', 'OE_ACTION_TOUCHPOINT')]:
        _cols = [c for c in _fw_order if c.endswith(f'_{_sfx}_{_w}') and not c.startswith('OE_IND_')]
        if _cols:
            feat_wide = feat_wide.withColumn(f'{_name}_{_w}',
                                             reduce(lambda a, b: a + b, [F.col(c) for c in _cols]))

feat_wide.write.mode('overwrite').parquet(f"{INTERMEDIATE_DIR}/feat_wide.parquet")
feat_wide_spk = spark.read.parquet(f"{INTERMEDIATE_DIR}/feat_wide.parquet")
print(f"Feature matrix: {len(feat_wide_spk.columns) - 2} feature cols")

# ── Missing-data flags per cohort ───────────────────────────────────────
_coh = (feat_wide_spk.groupBy('COHORT').agg(F.count('CUSTOMER_ID').alias('N_HCPs')).toPandas())
_coh = cohorts_df[['COHORT']].merge(_coh, on='COHORT', how='left').fillna({'N_HCPs': 0})
missing_data_flags = _coh[_coh['N_HCPs'] == 0]['COHORT'].tolist()
print("HCPs with NPP activity per cohort:")
print(_coh.to_string(index=False))
if missing_data_flags:
    print(f"Warning: Cohorts with ZERO NPP activity: {missing_data_flags}")
else:
    print("All cohorts have NPP activity.")

In [0]:
# ── Map CUSTOMER_ID (BH_ID) → PROVIDER_ID ───────────────────────────────────
feat_prov_spk = feat_wide_spk.join(
    BH_ID_MAP.withColumnRenamed('BH_ID','CUSTOMER_ID'), on='CUSTOMER_ID', how='left')

_n_unmapped = feat_prov_spk.filter(F.col('PROVIDER_ID').isNull()).count()
if _n_unmapped > 0:
    print(f"Warning: {_n_unmapped:,} (CUSTOMER_ID, COHORT) pairs with BH_ID not mapped - dropped")
feat_prov_spk = feat_prov_spk.filter(F.col('PROVIDER_ID').isNotNull())

# ── Full grid: 294,727 HCPs x 28 cohorts = 8.25M rows ──────────────────────
_hcp_spk    = hcp_spark.select('PROVIDER_ID').distinct()
_cohort_spk = spark.createDataFrame(cohorts_df[['COHORT','DATASET']])
grid_spk    = _hcp_spk.crossJoin(_cohort_spk)

# Feature columns (all except key identifiers)
FEAT_COLS = [c for c in feat_wide_spk.columns if c not in ('CUSTOMER_ID','COHORT')]

# ── Left join: NULLs for HCPs with no NPP activity, then fill 0 ──────────────
out_spk = (grid_spk
           .join(feat_prov_spk.select('PROVIDER_ID','COHORT', *FEAT_COLS)
                               .dropDuplicates(['PROVIDER_ID','COHORT']),
                 on=['PROVIDER_ID','COHORT'], how='left')
           .fillna(0.0, subset=FEAT_COLS)
           .withColumn('HCP_COHORT_ID', F.concat_ws('_', F.col('PROVIDER_ID'), F.col('COHORT'))))

n_rows = out_spk.count()
print(f"Output: {n_rows:,} rows x {len(out_spk.columns)} cols")
print(f"  PROVIDER_IDs : {out_spk.select('PROVIDER_ID').distinct().count():,}")
print(f"  Cohorts      : {out_spk.select('COHORT').distinct().count()}")
print(f"  Rows with any NPP (first feat > 0): "
      f"{out_spk.filter(F.col(FEAT_COLS[0]) > 0).count():,}")

In [0]:
qc = []
def check(label, passed, detail=""):
    qc.append((label, "PASS" if passed else "FAIL", str(detail)))

# 1. Row count = universe x cohorts
_expected = N_HCP * len(cohorts_df)
check("Rows = universe x cohorts", n_rows == _expected, f"{n_rows:,} vs {_expected:,}")

# 2. HCP_COHORT_ID unique
_n_id = out_spk.select('HCP_COHORT_ID').distinct().count()
check("HCP_COHORT_ID unique", _n_id == n_rows, f"{_n_id:,}")

# 3. No nulls in feature cols (first 5)
_nc = FEAT_COLS[:5]
_null_pd = out_spk.agg(*[F.sum(F.col(c).isNull().cast('int')).alias(c) for c in _nc]).toPandas()
_has_null = {c: int(_null_pd[c].iloc[0]) for c in _nc if int(_null_pd[c].iloc[0]) > 0}
check("No NULLs in feature cols (first 5)", len(_has_null) == 0, _has_null or "")

# 4. No negative values (first 5)
_neg_pd = out_spk.agg(*[F.sum((F.col(c) < 0).cast('int')).alias(c) for c in _nc]).toPandas()
_has_neg = {c: int(_neg_pd[c].iloc[0]) for c in _nc if int(_neg_pd[c].iloc[0]) > 0}
check("No negative feature values (first 5)", len(_has_neg) == 0, _has_neg or "")

# 5. Window nesting: P3 <= P6 <= P9 <= P12 (exposure touchpoint total)
_tp = 'OE_EXPOSURE_TOUCHPOINT'
_n_bad = out_spk.filter((F.col(f'{_tp}_P3') > F.col(f'{_tp}_P6')) |
                        (F.col(f'{_tp}_P6') > F.col(f'{_tp}_P9')) |
                        (F.col(f'{_tp}_P9') > F.col(f'{_tp}_P12'))).count()
check("Window nesting P3<=P6<=P9<=P12", _n_bad == 0, f"{_n_bad} violations")

# 5b. Only P3/P6/P9/P12 windows present
_bad_win = [c for c in FEAT_COLS if not c.endswith(('_P3','_P6','_P9','_P12'))]
check("Only P3/P6/P9/P12 features", len(_bad_win) == 0, _bad_win[:5] or "")

# 6. No cohorts with zero activity
check("No cohorts with zero NPP data", len(missing_data_flags) == 0,
      f"Empty: {missing_data_flags}" if missing_data_flags else "")

# 7. Indication features present
_ind = [c for c in FEAT_COLS if 'OE_IND' in c]
check("Indication features present", len(_ind) > 0, f"{len(_ind)} cols: {_ind[:4]}")

# 8. Content features present
_ctn = [c for c in FEAT_COLS if '_COUNT_' in c]
check("Content-category features present", len(_ctn) > 0, f"{len(_ctn)} cols")

# 9. Content tagging file
check("Content tagging file found", _tagg_ok,
      f"Missing: {CONTENT_TAG_FILE}" if not _tagg_ok else "")

# 10. I/E features present (from mdb_prep_config 'Metric Mappings')
_ie = [c for c in FEAT_COLS if c.endswith('_I_P3') or c.endswith('_E_P3')]
check("I/E features present", len(_ie) > 0, f"{len(_ie)} P3 I/E cols")

# 11. Excluded vendors absent (SFMC HQ Email lives in notebook 15)
_excl = [c for c in FEAT_COLS if any(c.startswith(f'OE_{v}_') for v in EXCLUDE_VENDORS)]
check("Excluded vendor features absent (SFMC)", len(_excl) == 0, _excl[:5] or "")

# 11b. Excluded content-tag features absent (SAMPLES)
_excl_ct = [c for c in FEAT_COLS for t in EXCLUDE_CONTENT_TAGS if c.startswith(f'OE_{t}_COUNT_')]
check("Excluded content features absent (SAMPLES)", len(_excl_ct) == 0, _excl_ct[:5] or "")

# 12. Engagement <= exposure and action <= exposure, for every channel, indication,
#     touchpoint total and window
_pairs = []
for c in FEAT_COLS:
    for _sfx in ('_I_', '_E_'):
        if _sfx in c:
            _r = c.replace(_sfx, '_R_')
            if _r in FEAT_COLS:
                _pairs.append((c, _r))
for _w in WINDOWS:
    for _n in ('ENGAGEMENT', 'ACTION'):
        if f'OE_{_n}_TOUCHPOINT_{_w}' in FEAT_COLS:
            _pairs.append((f'OE_{_n}_TOUCHPOINT_{_w}', f'OE_EXPOSURE_TOUCHPOINT_{_w}'))
_n_bad = out_spk.filter(reduce(lambda a, b: a | b, [F.col(a) > F.col(b) for a, b in _pairs])).count()
check("Engagement/action <= exposure (all cols)", _n_bad == 0, f"{len(_pairs)} pairs | {_n_bad:,} rows violate")

display(pd.DataFrame(qc, columns=['CHECK','RESULT','DETAIL']))

In [0]:
out_spk.write.mode("overwrite").option("overwriteSchema","true").parquet(OUTPUT_PATH)
print(f"Written {n_rows:,} rows -> {OUTPUT_PATH}")

In [0]:
# Preview
print(f"Shape: {n_rows:,} rows x {len(out_spk.columns)} cols | {len(FEAT_COLS)} feature cols")
display(out_spk.limit(3).toPandas())

# Fill rates (% rows with value > 0) computed in Spark
_fill_agg = out_spk.agg(
    *[F.sum((F.col(c) > 0).cast('int')).alias(c) for c in FEAT_COLS]).toPandas()
fill_df = pd.DataFrame(
    [(c, round(100.0 * int(_fill_agg[c].iloc[0]) / n_rows, 1)) for c in FEAT_COLS],
    columns=['FEATURE','FILL_RATE_PCT']
).sort_values('FILL_RATE_PCT', ascending=False)
display(fill_df)

## Notes for combine step

### Join key
`HCP_COHORT_ID = PROVIDER_ID + "_" + COHORT`, the same convention as the other NBA 3.1 feature notebooks. The output has 8,252,356 rows (294,727 HCPs x 28 cohorts), one row per `HCP_COHORT_ID`.

### NULL handling
* All 316 feature columns are 0-filled (no NULLs, no negative values). Do not impute again.
* A row of zeros means "no NPP activity in that window", except in cohorts before the data starts (below).

### Cohorts without data
After the 7-day promo lag, NPP events run from 2024-11-19 to 2026-09-17 (periods 202412 to 202609).

| Cohorts | What it means | Suggested handling |
| --- | --- | --- |
| c1 to c6 (lookback end 2024-06-17 to 2024-11-17) | All features are 0 because the data starts later | Exclude from training, or add a history indicator |
| c7 to c9 (end 2024-12-17 to 2025-02-17) | Only 1 to 3 periods of history, so P6/P9/P12 equal P3 | Treat P6/P9/P12 with care |
| c10 to c28 | Windows partly or fully covered | Use as is |

Share of HCPs with any exposure in P12: 16.9% (c7), 25.7% (c10), 33.7% (c16), 43.8% (c24), 48.3% (c28). Share with any engagement in P12: 1.5% (c7) to 26.2% (c28).

### Overlap with other notebooks
* SFMC HQ Email is excluded here, so there is no overlap with notebook 15 (HQ Emails).
* Indication columns (`OE_IND_*`) overlap each other (one record can count for several indications) and also appear in the channel totals. Do not sum them with channel columns.

### QC result of the full run (job run 358185454367814)
* Engagement and action never exceed exposure: 0 violating rows across 144 column pairs (every channel and indication I/E column against its R column, plus the engagement and action touchpoint totals, in all 4 windows).
* "No cohorts with zero NPP data" is expected to FAIL for c1 to c6 (see above).

### QC flags to watch
| Flag | What to check |
| --- | --- |
| `Content tagging file found = FAIL` | `Content_tagging_6_5.xlsx` is missing; all content would become `GENERAL`. |
| `I/E features present = FAIL` | The `Metric Mappings` sheet in `mdb_prep_config.xlsx` is missing or changed. |
| A supplemental Medscape channel disappears | `EXTRA_METRIC_MAPPINGS` in the Config cell was changed, or Medscape renamed its metrics. |
| `Engagement/action <= exposure (all cols) = FAIL` | The one-engagement-per-exposure rule in `npp_processing()` was changed or bypassed. |
| `No cohorts with zero NPP data = FAIL` | Expected in the current run: it lists c1 to c6, which fall before the data starts. Investigate only if any other cohort appears. |
| New channels or tags appear | Column names come from the data, so a new channel or tag adds columns. Check the combine schema. |

### Time windows
| Window | Event dates for a cohort with lookback end E |
| --- | --- |
| P3 | E minus 3 months (exclusive) to E (inclusive) |
| P6 | E minus 6 months to E |
| P9 | E minus 9 months to E |
| P12 | E minus 12 months to E |

## Feature dictionary and business rules

**Output:** `notebook_output/oe_nba_3_1/kerendia_nba_3_1_oe_promotional_features.parquet` in the NBA 3.1 volume.
**Grain:** one row per `PROVIDER_ID` x `COHORT` (294,727 HCPs x 28 cohorts = 8,252,356 rows).
**Columns:** 320 = 4 keys (`PROVIDER_ID`, `COHORT`, `DATASET`, `HCP_COHORT_ID`) + 316 features (79 base features x 4 windows).

### 1. Feature families

| Family | Column pattern | Base features | Columns | Definition |
| --- | --- | --- | --- | --- |
| Channel reach | `OE_{CHANNEL}_R_{W}` | 22 | 88 | Exposed interactions (category R in the metric mapping) on that channel, counted by exposure date |
| Channel engagement | `OE_{CHANNEL}_I_{W}` | 15 | 60 | Exposed interactions with at least one engagement (category I), counted in the exposure's period; always <= R |
| Channel action | `OE_{CHANNEL}_E_{W}` | 13 | 52 | Exposed interactions with at least one action (category E), counted in the exposure's period; always <= R |
| Indication | `OE_IND_{HF, CKD, T2D}_{R, I, E}_{W}` | 9 | 36 | Same counts, restricted to records whose `CF_INDICATION` contains the indication |
| Content category | `OE_{CONTENT_TAG}_COUNT_{W}` | 17 | 68 | Exposures whose content was tagged with that message category |
| Touchpoint totals | `OE_{EXPOSURE, ENGAGEMENT, ACTION}_TOUCHPOINT_{W}` | 3 | 12 | Sum of the channel R, I or E columns for the window (indication columns excluded) |
| **Total** | | **79** | **316** | |

W is one of P3, P6, P9, P12.

**Channels and the flags they have:**

| Channel | R | I | E |
| --- | --- | --- | --- |
| DOXIMITY_ANNOUNCEMENT, DOXIMITY_DOCNEWS, DOXIMITY_DOCPOINT, DOXIMITY_DOCSPOT, DOXIMITY_FORMULARYONE | Yes | Yes | Yes |
| MEDSCAPE_3RD_PARTY_EMAIL, MEDSCAPE_BANNER, MEDSCAPE_DDTRD, MEDSCAPE_FAP, MEDSCAPE_FFQ, MEDSCAPE_PULSEPOINT_BANNER, MEDSCAPE_RECAP | Yes | Yes | Yes |
| MEDSCAPE_BRANDALERT, PEERDIRECT_MD_MINUTE_EMAIL, PULSEPOINT_BANNER | Yes | Yes | |
| MEDSCAPE_BRANDSPOT | Yes | | Yes |
| MEDSCAPE_EHR, PEERDIRECT_MDFI_BANNER, PEERDIRECT_MDFI_EMAIL, PEERDIRECT_MDFI_WEB, PEERDIRECT_MD_MINUTE_BANNER, PEERDIRECT_MD_MINUTE_WEB | Yes | | |

**Content tags (17):** ACCESS_COVERAGE, BASIC_PRODUCT_INFO, COST, DISEASE_AWARENESS_MANAGEMENT, DOSING_ADMINISTRATION, EFFICACY, EVENT, GENERAL, GUIDELINES, MECHANISM_OF_ACTION, MECHANISM_OF_DISEASE, PATIENT_ADHERENCE_SUPPORT, PATIENT_PROFILE, SAFETY_EFFICACY, SOCIAL_PROOF, STUDY_DESIGN, UNMET_NEED.

Channel and tag lists come from the data, so new values in the NPP table create new columns.

### 2. Time windows

Every `lookback_end_date` falls on the 17th. Each event (after the promo lag) is assigned to an 18th-to-17th period; for example 2026-08-18 to 2026-09-17 is period 202609. Pn is the n periods ending at the cohort's `lookback_end_date`.

| Window | Example: c28 (lookback 2025-09-17 to 2026-09-17) |
| --- | --- |
| P3 | 2026-06-18 to 2026-09-17 |
| P6 | 2026-03-18 to 2026-09-17 |
| P9 | 2025-12-18 to 2026-09-17 |
| P12 | 2025-09-18 to 2026-09-17 |

The windows are nested (P3 is inside P6, and so on), so each column is greater than or equal to the shorter window's column. Nothing after `lookback_end_date` is used, so there is no leakage into the prediction month.

### 3. Business rules applied

1. **Source:** `kerendia_oe_journey.oe_kerendia_fact_npp_inference` (non-personal promotion only; calls and events are out of scope).
2. **Product filter:** `PROD_BRAND_NM = 'KERENDIA'` and `BAYER_HCP_ID` not null.
3. **Excluded vendor:** SFMC (HQ Email) is removed when the data is loaded, because notebook 15 already builds HQ email features. It is excluded from every family, including content, indication and touchpoint totals.
4. **HCP universe and ID mapping:** NPP `CUST_ID` (a BH_ID) is mapped to the NBA 3.1 `PROVIDER_ID` with `kerendia_nba_3.SOURCE_PROVIDER_BHID_NPI_MAPPING`. Newer BH_IDs are mapped through `kerendia_oe_npi_bhid_mapping_inference` (BH_ID to 10-digit NPI to PROVIDER_ID). If a BH_ID maps to several PROVIDER_IDs, the smallest is kept, so each event is counted once. Only HCPs in `HCP_Universe_without_call_hcps` are kept.
5. **Source dedup:** one record per `UNIQ_NPP_ID, DATE_ID, CUST_ID, VEND_NM, CHNL_NM, MTR_NM`.
6. **Channel:** vendor + sub-channel. The sub-channel comes from `TACT_NM` patterns (Doximity DocNews, DocSpot, DocPoint, FormularyOne, Announcement; Medscape tactic codes EA, BA, BS, PP, FFQ, RECAP, TLM, PPEHR, FAP, DDTRD; PeerDirect MDFI and MD Minute placements; PulsePoint Banner). It is then uppercased and standardised.
7. **Reach / engagement / action flags:** from the `Metric Mappings` sheet of `helper_files/mdb_prep_config.xlsx`. Category R = reach (exposure), I = engagement, E = action. Records are matched on the standardised channel (`Event Type`) and metric name (`Mapped Metric`). A pair listed under several categories gets each flag. Records whose pair is not in the sheet are dropped. The sheet is extended at load time with `EXTRA_METRIC_MAPPINGS` (Config cell) for MEDSCAPE_BANNER, MEDSCAPE_PULSEPOINT_BANNER, MEDSCAPE_BRANDSPOT, MEDSCAPE_DDTRD and MEDSCAPE_FAP, whose data uses generic metric names (EXPOSURE, IMPRESSIONS, DELIVERED, OPENED, VIEW, LINK CLICK, CLICKED, ...). The Excel file itself is not changed.
8. **Engagement never exceeds exposure:** each interaction (HCP x channel x content x `UNIQ_NPP_ID`) is one exposure, dated on its first exposure record. It counts at most one engagement and one action, however many open, view or click records it has. An email opened 3 times in one day, or on several days, is one engagement. Engagement and action are counted in the exposure's period, so I <= R and E <= R in every window. Engagement or action records with no exposure for the interaction are dropped.
9. **Content tag:** a left join to `Content_tagging_6_5.xlsx` on 14 creative and campaign attributes. Untagged, `-` or `Not Needed` become `GENERAL`. The `SAMPLES` tag columns are dropped; those exposures still count in channel, indication and touchpoint totals.
10. **Promo lag:** every event date is moved forward 7 days before its period is assigned.
11. **Event dedup:** one row per HCP x channel x content x interaction; each row counts as 1.
12. **Indication:** `CF_INDICATION` is uppercased and matched by substring. `HFPEF,CKD IN T2D` counts toward HF, CKD and T2D, so indication columns overlap and should not be summed. `HF` also matches `HFPEF`. There are no T1D records, so no T1D columns. UNKNOWN or N/A records count only in channel and content features.
13. **Fill rule:** HCP x cohort rows with no NPP activity are 0. All features are counts.

### 4. Validation of the full run

Checked directly on the output file:
* 8,252,356 rows, one per `HCP_COHORT_ID`; 294,727 HCPs; 28 cohorts.
* 0 NULLs and 0 negative values across all 316 features.
* Window nesting holds for all 79 base features (0 violations).
* No SFMC or SAMPLES columns.
* Engagement and action never exceed exposure: 0 violating rows across 144 column pairs.
* Touchpoint totals equal the sum of their channel columns (0 mismatches in P12).

### 5. Known limitations

* **Data starts 2024-11-19 (after the lag).** Cohorts c1 to c6 are all zero, and c7 to c9 only have 1 to 3 periods of history. See the combine notes.
* **Supplemental Medscape mappings are an assumption.** They follow the sheet's conventions but were not supplied by the business; confirm them with the OE owner.
* **MEDSCAPE_BRANDSPOT has no engagement column:** none of its engagement records were attached to an exposed interaction. The reason has not been investigated.
* **Some Medscape records are still dropped:** records whose `TACT_NM` gives no sub-channel (VIEW, MEDIA TIME SPENT, PARTICIPATION, 2024-11 to 2025-01) do not match any channel rule.
* **Engagement or action without an exposure is not counted** (one engagement or action per exposed interaction).
* **Engagement timing:** an engagement is counted in its exposure's period, even if the open happened a few days later (for example after the period's cutoff on the 17th).